In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

# 1. 패키지

In [ ]:
# python-dotenv, langchain, langchain-openai, langchain-pinecone, langchin-community, docx2txt
# langchain-upstage, langchain-ollama
# pandas

In [21]:
%pip install -q pandas==2.2.3

Note: you may need to restart the kernel to use updated packages.


# 2. 환경변수 설정

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
OPENAI_LLM_MODEL = "gpt-4o-mini"
UPSTAGE_LLM_MODEL = "solar-pro4"
# OPENAI_EMBEDDING_MODEL = "text-embedding-3-large" # 차원수 3072
UPSTAGE_EMBEDDING_MODEL = "embedding-query" # 차원수 4096

PINECONE_INDEX_NAME = "better-rag-index"
PINECONE_INDEX_METRIC = 'cosine'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_DIMENSION = 4096

# 3. 문서를 chunk로 분할하기

In [4]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = Docx2txtLoader('tax/소득세법_with_markdown.docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=100, 
    separators=['\n\n', '\n', ' ', '']
)

document = loader.load()
documents = text_splitter.split_documents(document)
print(f"분리 결과 : {len(documents)} 청크")

분리 결과 : 184 청크


# 4. metadata 추가하기
- chunk 내용의 title, category, 조항

In [5]:
import re
def remove_special_chars(text:str) -> str:
    "특수문자 및 \n제거 (:, .는 그대로)"
    text = text.replace("\n", " ")
    # 한글, 영문, 숫자, 공백, . , : |-만 남기고 전부 제거
    cleaned = re.sub(r"[^가-힣a-zA-Z0-9\s\.,:|\-]", "", text).strip()
    # 불필요한 중복 공백을 제거
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned
content = " ▶️ **소득세 납세 의무 ** : \n\n|이 법 또는 「법인세법」에 따라 소득에 대한 소득세를 부과  "
remove_special_chars(content)

'소득세 납세 의무 : |이 법 또는 법인세법에 따라 소득에 대한 소득세를 부과'

In [6]:
%%time
# 제목을 추출하는 함수(사용 llm은 exaone3.5:2.4b) ollama list -> ollama pull exaone3.5:2.4b
from langchain_ollama import ChatOllama
from langchain_upstage import ChatUpstage
def extract_title_with_llm(content):
    "content의 제목을 반환"
    # title_extractor_llm = ChatOllama(
    #     model = 'exaone3.5:2.4b',
    #     temperature=0.1
    # )
    title_extractor_llm = ChatUpstage(
        model = UPSTAGE_LLM_MODEL,
        temperature=0.1
    )
    prompt = f"""다음 소득세법 조문의 핵심 제목을 100토큰 이내로 완벽하게 말이 되도록 추출해 주세요.
    문장이 끊기지 않도록 해주세요.
    조문 : {content}"""
    ai_message = title_extractor_llm.invoke(prompt)
    title = ai_message.content.strip()
    return remove_special_chars(title)
extract_title_with_llm(remove_special_chars(documents[43].page_content))

CPU times: total: 5.39 s
Wall time: 9.09 s


'공동사업에 대한 소득공제 등 특례와 세율 및 세액공제'

## 4.1 카테고리 분류

In [7]:
def categorize_content(content):
    "content의 카테고리를 list로 반환"
    # gpt가 meta데이터 활용한다고 하니 만들어준 카테고리
    categories = {
    '납세의무': ['납세의무', '거주자', '비거주자', '원천징수', '공동사업자', '상속인', '증여자', '신탁재산'],
    '세율계산': ['세율', '과세표준', '산출세액', '결정세액', '종합소득', '퇴직소득'],
    '근로소득': ['근로소득', '총급여', '급여', '연봉', '임금', '퇴직소득'],
    '사업소득': ['사업소득', '공동사업', '주택임대소득', '부업소득'],
    '이자배당': ['이자소득', '배당소득', '예금이자', '채권', '의제배당'],
    '양도소득': ['양도소득', '자산양도', '부동산양도', '주식양도'],
    '연금소득': ['연금소득', '공적연금', '사적연금', '연금보험'],
    '기타소득': ['기타소득', '상금', '보상금', '발명보상금', '종교인소득'],
    '공제감면': ['공제', '소득공제', '세액공제', '기본공제', '감면'],
    '비과세': ['비과세', '면제', '복무급여', '실업급여', '출산휴가급여', '장학금'],
    '신고납부': ['신고', '납부', '납세지', '신고기한', '원천징수'],
    '과세기간': ['과세기간', '과세연도', '사업연도'],
    '과세소득구분': ['종합소득', '퇴직소득', '양도소득', '금융투자소득'],
    }
    result = [] # 각 카테고리를 추가
    for category, keywords in categories.items():
        #print(category, keywords)
        # print(any([keyword in content for keyword in keywords])) : 하나라도 True인지 여부
        if any([keyword in content for keyword in keywords]):
            result.append(category)
    if not result:
        result.append("기타")
    return result

In [8]:
print(categorize_content(documents[43].page_content))
print(categorize_content("아무 내용도 없다"))

['납세의무', '세율계산', '근로소득', '사업소득', '공제감면', '과세기간', '과세소득구분']
['기타']


- 개선된 카테고리 분류방법 : 키워드의 중요도에 따라 가중치를 부여하고, 가중치가 높은 순으로 카테고리를 반환

In [9]:
def get_category():
    return {
        '납세의무': {
            '납세의무': 3, '거주자': 3, '비거주자': 3, '납세의무자': 3, 
            '원천징수': 2, '원천징수의무자': 2, '공동사업자': 2, 
            '상속인': 2, '증여자': 2, '신탁재산': 2
        },

        '세율계산': {
            '세율': 3, '소득세': 3, '과세표준': 3, '산출세액': 3, '세액': 3,
            '결정세액': 2, '세액계산': 2, '기본세율': 2, '세율적용': 2,
            '누진세율': 2, '종합소득세': 2
        },

        '근로소득': {
            '근로소득': 3, '총급여': 3, '급여': 3, '연봉': 3, '임금': 3,
            '근로소득금액': 2, '총급여액': 2, '상여': 2, '수당': 2,
            '봉급': 2, '직장인': 2
        },

        '사업소득': {
            '사업소득': 3, '총수입금액': 3, '필요경비': 3,
            '사업자': 2, '사업소득금액': 2, '결손금': 2, '이월결손금': 2,
            '주택임대소득': 2, '공동사업': 2
        },

        '이자배당': {
            '이자소득': 3, '배당소득': 3, '예금이자': 2, '채권': 2,
            '의제배당': 2, '배당세액공제': 2, '분리과세이자소득': 2,
            '분리과세배당소득': 2
        },

        '양도소득': {
            '양도소득': 3, '자산양도': 2, '부동산양도': 2, '주식양도': 2,
            '양도차익': 2, '취득가액': 2, '양도가액': 2, '양도소득금액': 2
        },

        '연금소득': {
            '연금소득': 3, '연금계좌': 3, '연금저축': 3,
            '퇴직연금': 2, '공적연금': 2, '사적연금': 2, '연금보험': 2,
            '연금수령': 2
        },

        '기타소득': {
            '기타소득': 3, '가상자산': 3, '가상자산소득': 2,
            '상금': 2, '보상금': 2, '종교인소득': 2, '원고료': 2,
            '복권': 1, '당첨금': 1, '발명보상금': 1
        },

        '공제감면': {
            '공제': 3, '소득공제': 3, '세액공제': 3,
            '기본공제': 2, '인적공제': 2, '특별공제': 2, '추가공제': 2,
            '근로소득공제': 2, '연금소득공제': 2, '퇴직소득공제': 2,
            '연금계좌세액공제': 2, '감면': 2
        },

        '비과세': {
            '비과세': 3, '비과세소득': 3, '면제': 2, '세액감면': 2,
            '소득세면제': 2, '복무급여': 1, '실업급여': 1, 
            '출산휴가급여': 1, '장학금': 1
        },

        '신고납부': {
            '신고': 3, '확정신고': 3, '과세표준확정신고': 3, 
            '납부': 3, '중간예납': 2, '가산세': 2,
            '신고기한': 2, '납부기한': 2, '납세지': 2
        },

        '과세기간': {
            '과세기간': 3, '과세연도': 3, '사업연도': 2, 
            '과세기간종료일': 2
        },

        '과세방식': {
            '종합과세': 3, '분리과세': 3, '합산과세': 2, 
            '종합소득과세표준': 2, '분리과세소득': 2, 
            '금융투자소득': 2
        },

        '장부기장': {
            '장부': 3, '복식부기': 3, '간편장부': 2, '기장': 2,
            '장부기록': 2, '증명서류': 2, '기장세액공제': 2
        }
    }

In [10]:
def categorize_content(content, top_k=None):
    """
    content의 카테고리를 분류 - 점수 기반으로 모든 카테고리를 점수순으로 list를 반환(top_k만큼)
        Parameters :
        - content : 카테고리를 분류할 텍스트
        - top_k : 상위 몇개까지의 카테고리를 반환할지 정수나 None(None이면 모든 카테고리를 반환)
        Returns :
        - 카테고리 리스트(점수순) ex.{'장부기장':30, '과세방식':45}=>['과세방식','장부기장']
    """
    category_keywords = get_category()
    category_scores = {}
    # 각 카테고리별 점수 계산
    for category, weighted_keywords in category_keywords.items():
        # print(category, weighted_keywords)
        score = 0
        for keyword, weight in weighted_keywords.items():
            # print(keyword, weight)
            count = content.count(keyword) # content에 keyword가 나오는 횟수
            score += count * weight
        if score>0:
            category_scores[category] = score
    # print(category_scores)
    sorted_categories = sorted(category_scores.items(), key=lambda item:item[1], reverse=True)#내림차순
    # print(sorted_categories)
    all_categories = [category[0] for category in sorted_categories]
    # 매칭되는 카테고리가 없을 경우 ["기타"]를 반환
    if not all_categories:
        all_categories = ["기타"]
    # print(all_categories)
    if top_k is not None: #if top_k:
        return all_categories[:top_k]
    else:
        return all_categories

In [11]:
content = """1. 원천징수하는 자가 거주자인 경우: 그 거주자의 주된 사업장 소재지. 다만, 주된 사업장 외의 사업장에서 원천징수를 하는 경우에는 그 사업장의 소재지, 사업장이 없는 경우에는 그 거주자의 주소지 또는 거소지로 한다.
2. 원천징수하는 자가 비거주자인 경우: 그 비거주자의 주된 국내사업장 소재지. 다만, 주된 국내사업장 외의 국내사업장에서 원천징수를 하는 경우에는 그 국내사업장의 소재지, 국내사업장이 없는 경우에는 그 비거주자의 거류지(居留地) 또는 체류지로 한다.
3. 원천징수하는 자가 법인인 경우: 그 법인의 본점 또는 주사무소의 소재지
장부 장부 장부 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기
"""
#content = "카테고리의 아무 단어도 없네"
print(categorize_content(content))
print(categorize_content("연봉 5천만원인 직장인의 소득세는 얼마예요?"))
print(categorize_content(documents[43].page_content))

['장부기장', '납세의무']
['근로소득', '세율계산']
['세율계산', '공제감면', '사업소득', '납세의무', '과세방식', '과세기간']


## 4.2 조항 추출하기

In [12]:
# content에서 제000조 스타일만 추출 : "「9조,60조」"
import re
article_match = re.findall(r"제(\d{1,3}조)", "제60조 + 기타내용")
article = list(set(article_match))
# article = sorted(article, key=lambda x : int(x[:-1]))
article.sort(key=lambda x : int(x[:-1]))
"「" + ", ".join(article) + "」"

'「60조」'

In [13]:
a = '60조'
int(a[:-1])

60

In [14]:
import re
def get_article(content):
    """
    content내의 조항들을 반환
    Parameter :
    - content : 조항을 추출한 text
    Returns : content내에 등장된 조항 수들 문자로 반환 ex."「9조,60조」"
    """
    article_match = re.findall(r"제(\d{1,3}조)", content)
    article_list = list(set(article_match))
    article_list.sort(key=lambda x : int(x[:-1]))
    if article_list:
        return "「" + ", ".join(article_list) + "」"
    return ""

In [15]:
print(get_article(content))
print(get_article(documents[43].page_content))


「43조, 51조, 54조, 55조, 59조」


In [16]:
# documents
metadata={'source': 'tax_docx/소득세법_with_markdown.docx'}
metadata['title'] = '제목'
metadata['category'] = ['납세의무','소득공제']
metadata['article'] = '「43조, 51조, 54조, 55조, 59조」'
metadata

{'source': 'tax_docx/소득세법_with_markdown.docx',
 'title': '제목',
 'category': ['납세의무', '소득공제'],
 'article': '「43조, 51조, 54조, 55조, 59조」'}

In [19]:
%%time
# documents의 metadata추가
from langchain_core.documents import Document
enhanced_chunks = [] # metadata를 추가한 청크들(documents)
for i, chunk in enumerate(documents[:15]):
    if i%5==0:
        print(f"진행 : {i:2} / 15 : {i/15*100:.1f}")
    
    content = chunk.page_content # str
    metadata = chunk.metadata.copy()    # dict
    metadata['title'] = extract_title_with_llm(remove_special_chars(content)) # 제목 추출
    metadata['category'] = categorize_content(content) # 카테고리 추출
    metadata['chunk_id'] = f'chunk_{i:03d}'
    # 조문 추출 : "" + "「0조」"
    article = get_article(content)
    if article:
        metadata['article'] = article # 추출된 조항이 있을 경우만 metadata에 추가
    enhanced_chunks.append(Document(page_content=content,
                                   metadata=metadata)) 
print(f"{len(enhanced_chunks)} 청크 전처리 완료")

진행 :  0 / 15 : 0.0
진행 :  5 / 15 : 33.3
진행 : 10 / 15 : 66.7
15 청크 전처리 완료
CPU times: total: 219 ms
Wall time: 13.4 s


In [22]:
import pandas as pd
# enhanced_chunks → dict list → dataframe (csv 백업)
data = []
for chunk in enhanced_chunks:
    row = {"page_content":chunk.page_content}
    row.update(chunk.metadata)
    data.append(row)
df = pd.DataFrame(data)
df.to_csv("./tax/소득세법enhanced_chunks일부.csv", index=False, encoding='cp949')

## 4.3 백업받은 csv를 읽어오기

In [24]:
import pandas as pd
df = pd.read_csv("./tax/소득세법enhanced_chunks.csv", encoding='cp949')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   page_content  184 non-null    object
 1   source        184 non-null    object
 2   title         184 non-null    object
 3   category      184 non-null    object
 4   chunk_id      184 non-null    object
 5   article       181 non-null    object
dtypes: object(6)
memory usage: 8.8+ KB


In [25]:
# 문자열("['납세의무', '세율계산']") => 리스트(['납세의무', '세율계산'])
import ast
category = "['납세의무', '세율계산']"

# 리스트로 변환
category_lst = ast.literal_eval(category)
print(category_lst, type(category_lst))

['납세의무', '세율계산'] <class 'list'>


In [26]:
# df['category'] 타입변경(문자열->리스트)
df['category'] = df['category'].apply(ast.literal_eval)
df.loc[0, 'category']

['납세의무', '세율계산', '신고납부', '사업소득', '이자배당', '근로소득', '양도소득', '기타소득']

## 4.4 DataFrame → Doclist → PineconeSave

In [27]:
for i, row in df.loc[0:1].iterrows():
    print(i)
    print(row)
    print(type(row))

0
page_content    소득세법\n\n\n\n소득세법\n\n[시행 2026. 7. 1.] [법률 제2122...
source                           tax_docx/소득세법_with_markdown.docx
title           소득세법 총칙 개인 소득에 따라 적정 과세하여 조세부담 형평과 재정수입 확보를 목적...
category         [납세의무, 세율계산, 신고납부, 사업소득, 이자배당, 근로소득, 양도소득, 기타소득]
chunk_id                                                chunk_000
article                                             「1조, 2조, 13조」
Name: 0, dtype: object
<class 'pandas.core.series.Series'>
1
page_content    1. 구성원 간 이익의 분배비율이 정하여져 있고 해당 구성원별로 이익의 분배비율이 ...
source                           tax_docx/소득세법_with_markdown.docx
title           소득세 및 법인세 납세의무 분배 - 구성원별 분배 확인 시: 구성원별로 소득세 또는...
category                     [납세의무, 세율계산, 사업소득, 양도소득, 신고납부, 과세방식]
chunk_id                                                chunk_001
article                 「1조, 2조, 14조, 43조, 44조, 101조, 119조, 127조」
Name: 1, dtype: object
<class 'pandas.core.series.Series'>


In [28]:
%%time
from langchain_core.documents import Document
import pandas as pd
enhanced_chunks = []
for i, row in df.iterrows():
    content = row['page_content']
    metadata = {
        'source': row['source'], # row.source
        'title':row['title'],
        'category':row['category'],
        'chunk_id' : row['chunk_id'],
    }
    # article은 결측치가 아닐 경우만 추가
    if pd.notna(row['article']):
        metadata['article'] = row['article']
    #print(metadata)
    # Document 객체 생성 및 리스트 추가
    doc = Document(page_content=content, metadata=metadata)
    enhanced_chunks.append(doc)
print(f"✔️ {len(enhanced_chunks)} 청크 처리 완료")
print(type(enhanced_chunks[0]))

✔️ 184 청크 처리 완료
<class 'langchain_core.documents.base.Document'>
CPU times: total: 31.2 ms
Wall time: 20.7 ms


# 5. 임베딩 모델 설정

In [29]:
from langchain_upstage import UpstageEmbeddings
embedding = UpstageEmbeddings(model = UPSTAGE_EMBEDDING_MODEL)
len(embedding.embed_query("차원 확인용"))

4096

# 6. Pinecone 인덱스 생성 및 vector DB(Store) 저장

In [30]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# pinecone 클라이언트 객체
pc = Pinecone()

# indexnames = pc.list_indexes().names()
# if PINECONE_INDEX_NAME not in indexnames:
dimension = len(embedding.embed_query("차원 확인용"))

if not pc.has_index(PINECONE_INDEX_NAME): #has_index():해당 인덱스가 있는지 여부
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=dimension,
        metric=PINECONE_INDEX_METRIC,
        spec = ServerlessSpec(region=PINECONE_INDEX_REGION, 
                             cloud=PINECONE_INDEX_CLOUD)
    )
    print(f'인덱스 생성 완료 : {PINECONE_INDEX_NAME}')

else:
    # pinecone vector DB에 chunk하나 추가 저장 (데이터 삭제 시, 에러 방지용)
    PineconeVectorStore.from_documents(
        documents=[enhanced_chunks[0]],
        embedding=embedding,
        index_name = PINECONE_INDEX_NAME
    )
    # index의 모든 데이터 지우기
    index = pc.Index(PINECONE_INDEX_NAME)
    index.delete(delete_all=True)
    # pc.delete_index(PINECONE_INDEX_NAME) : 인덱스 자체를 지울 때
    print(f'인덱스 존재 : {PINECONE_INDEX_NAME}')

C:\Users\Admin\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


인덱스 존재 : better-rag-index


In [31]:
# 빈 인덱스 확인
index = pc.Index(PINECONE_INDEX_NAME)
index.describe_index_stats()

{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

In [32]:
%%time
# pinecone vector DB 업로드
vector_database = PineconeVectorStore.from_documents(
    documents = enhanced_chunks,
    embedding = embedding,
    index_name = PINECONE_INDEX_NAME
)

CPU times: total: 12.5 s
Wall time: 23.5 s


In [ ]:
# 업로드한 vector DB load
vector_database = PineconeVectorStore(
    embedding=embedding,
    index_name=PINECONE_INDEX_NAME
)

In [33]:
# 인덱스확인
index = pc.Index(PINECONE_INDEX_NAME)
index.describe_index_stats()

{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 184}},
 'total_vector_count': 184,
 'vector_type': 'dense'}

# 7. 관련 문서 미리보기 (metadata 활용)

In [34]:
query = "연봉 5000만원인 직장인의 소득세는 얼마예요?"
query = "종합소득 5000만원인 근로소득이 있는 거주자의 소득세는 얼마예요?"
categorize_content(query)

['납세의무', '세율계산', '근로소득']

In [35]:
# retriever 생성(['납세의무', '세율계산', '근로소득'] 중 하나라도 포함된 문서에서 유사도 검색)
retriever = vector_database.as_retriever(
                search_type="similarity",
                search_kwargs = {
                    "k":4,
                    "filter":{"category" : {'$in':categorize_content(query)}}
                }
            )
docs = retriever.invoke(query)
print("검색된 관련 문서")
for i, doc in enumerate(docs):
    #id = doc.id
    chunk_id = doc.metadata['chunk_id']
    category = doc.metadata['category']
    article  = doc.metadata.get('article', '조항없음')
    title = doc.metadata.get('title') # reranking 작업 시 사용할 예정
    print(f'{i+1}. {chunk_id} : {article} {category} : {title}')

검색된 관련 문서
1. chunk_043 : 「43조, 51조, 54조, 55조, 59조」 ['세율계산', '공제감면', '사업소득', '납세의무', '과세방식', '과세기간'] : 소득세 계산 특례 및 세액공제 공동사업 소득공제 및 세액공제 적용 시 특수관계인의 지출이 주된 사업자의 합산 과세 대상에 포함되며, 해당 금액 한도 내에서 공제 가능. 세율 구조는 과세표준 구간별 차등 적용되며, 거주자의 종합소득 및 퇴직소득에 대한 세액 계산 방식 명시.
2. chunk_037 : 「22조, 47조, 48조, 49조, 50조」 ['공제감면', '근로소득', '납세의무', '과세기간', '연금소득', '과세방식'] : 소득세 공제 규정 요약 - 일용근로자 공제: 1일 최대 15만원 공제 2018년 개정. - 총급여액 공제: 총급여액이 공제액 이하인 경우 해당 금액 공제 근로소득공제. - 연금소득 공제: 연금소득 합산액에서 최대 900만원 공제 2013년 개정. - 퇴직소득 공제: 근속연수별 공제 후, 근속연수 미만 기간은 1년으로 환산 공제 2014년 개정. - 종합소득 공제: 자연인 기준 1인당 연 150만원 공제 2015년 개정.
3. chunk_048 : 「59조, 134조」 ['근로소득', '공제감면', '세율계산', '신고납부', '납세의무', '과세기간'] : 소득세법 주요 조항 요약: - 근로소득세액공제: 총급여액에 따라 차등 공제 최대 한도 명시. - 일용근로자 공제: 원천징수 시 추가 공제율 적용. - 자녀세액공제: 자녀 수 및 출산입양 여부에 따라 차등 공제 최대 한도 및 추가 혜택 포함. 이 요약은 핵심 내용을 100토큰 이내로 간결하게 전달합니다.
4. chunk_036 : 「4조, 16조, 45조, 46조, 47조, 133조, 156조」 ['공제감면', '이자배당', '납세의무', '근로소득', '과세기간', '사업소득', '세율계산', '신고납부', '연금소득', '과세방식'] : 소득공제 및 세액공제 특례 - 결손금 및 이월결손금 공제: 

In [36]:
[{'category':{'$eq':cat}} for cat in categorize_content(query)]

[{'category': {'$eq': '납세의무'}},
 {'category': {'$eq': '세율계산'}},
 {'category': {'$eq': '근로소득'}}]

In [37]:
# retriever 생성(['납세의무', '세율계산', '근로소득'] 문서에서 유사도 검색)
filter_condition = {
    '$and':[{'category':{'$eq':cat}} for cat in categorize_content(query)]
}

retriver = vector_database.as_retriever(
    search_type="similarity",
    search_kwargs = {
        'k':4,
        'filter':filter_condition
    }
)

docs = retriever.invoke(query)

print("검색된 관련 문서")
for i, doc in enumerate(docs):
    #id = doc.id
    chunk_id = doc.metadata['chunk_id']
    category = doc.metadata['category']
    article  = doc.metadata.get('article', '조항없음')
    title = doc.metadata.get('title') # reranking 작업시 사용할 예정
    print(f'{i+1}. {chunk_id} : {article} {category} : {title}')

검색된 관련 문서
1. chunk_043 : 「43조, 51조, 54조, 55조, 59조」 ['세율계산', '공제감면', '사업소득', '납세의무', '과세방식', '과세기간'] : 소득세 계산 특례 및 세액공제 공동사업 소득공제 및 세액공제 적용 시 특수관계인의 지출이 주된 사업자의 합산 과세 대상에 포함되며, 해당 금액 한도 내에서 공제 가능. 세율 구조는 과세표준 구간별 차등 적용되며, 거주자의 종합소득 및 퇴직소득에 대한 세액 계산 방식 명시.
2. chunk_037 : 「22조, 47조, 48조, 49조, 50조」 ['공제감면', '근로소득', '납세의무', '과세기간', '연금소득', '과세방식'] : 소득세 공제 규정 요약 - 일용근로자 공제: 1일 최대 15만원 공제 2018년 개정. - 총급여액 공제: 총급여액이 공제액 이하인 경우 해당 금액 공제 근로소득공제. - 연금소득 공제: 연금소득 합산액에서 최대 900만원 공제 2013년 개정. - 퇴직소득 공제: 근속연수별 공제 후, 근속연수 미만 기간은 1년으로 환산 공제 2014년 개정. - 종합소득 공제: 자연인 기준 1인당 연 150만원 공제 2015년 개정.
3. chunk_048 : 「59조, 134조」 ['근로소득', '공제감면', '세율계산', '신고납부', '납세의무', '과세기간'] : 소득세법 주요 조항 요약: - 근로소득세액공제: 총급여액에 따라 차등 공제 최대 한도 명시. - 일용근로자 공제: 원천징수 시 추가 공제율 적용. - 자녀세액공제: 자녀 수 및 출산입양 여부에 따라 차등 공제 최대 한도 및 추가 혜택 포함. 이 요약은 핵심 내용을 100토큰 이내로 간결하게 전달합니다.
4. chunk_036 : 「4조, 16조, 45조, 46조, 47조, 133조, 156조」 ['공제감면', '이자배당', '납세의무', '근로소득', '과세기간', '사업소득', '세율계산', '신고납부', '연금소득', '과세방식'] : 소득공제 및 세액공제 특례 - 결손금 및 이월결손금 공제: 

# 8. Langchain 답변 생성

In [38]:
from langchain_upstage import ChatUpstage
from langchain_upstage import UpstageEmbeddings
# from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

In [39]:
# 임베딩 & LLM 초기화
load_dotenv()
llm = ChatUpstage(
    model = 'solar-pro4',
    reasoning_effort='low'
)
embedding = UpstageEmbeddings(model=UPSTAGE_EMBEDDING_MODEL)

# vector DB load
# pc = Pinecone()
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=PINECONE_INDEX_NAME
)

In [40]:
# 키워드 사전 활용
keyword_dict  = [
    "사람을 나타내는 표현 -> 거주자",
    "직장인 -> 근로소득이 있는 거주자", 
    "월급쟁이 -> 근로소득이 있는 거주자",
    "회사원 -> 근로소득이 있는 거주자",
    "연봉 -> 종합소득",
    "월급 -> 근로소득",
    "세금 -> 소득세",
    "공제받다 -> 공제를 적용받다",
    "얼마나 내야하나 -> 세액은 얼마인가",
    "계산해줘 -> 계산하면 얼마인가"
]

prompt = ChatPromptTemplate.from_template(f"""사용자의 질문을 보고, 우리의 사전을 참고해서 사용자 질문을
변경해 주세요. 만약 변경할 필요가 없을 경우, 사용자의 질문을 변경하지 않아도 됩니다. 오직 질문만 리턴해 주세요.
사전 : {keyword_dict}
질문 : {{question}}""")

# llm.invoke(prompt.invoke({'question':"월급이 높은 남자가 있습니다"}))
keyword_chain = prompt | llm | StrOutputParser()
query = "연봉 5000만원인 회사원의 소득세는 얼마예요?"
keyword_chain.invoke({'question':query})
#keyword_chain.invoke(query)

'종합소득 5000만원인 근로소득이 있는 거주자의 세액은 얼마인가'

In [41]:
# Retriver 생성
retriever = vectorStore.as_retriever(
    search_type='similarity',
    search_kwargs = {
        'k':3,
        'filter' : {'category':{'$in':categorize_content(query)}}
    }
)


# 프롬프트 템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 
모른다고 답하세요.
질문 : {{query}}
문맥 : {{context}}
답변 :"""

prompt = ChatPromptTemplate.from_template(template)

# 검색된 documents를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

# RAG chain 구성 (LCEL 방식)
rag_chain = (
    {
        'context':retriever | format_documents,
        'query' : RunnablePassthrough() # 질문만 그대로 전달 가능
    }
    | prompt
    | llm
    | StrOutputParser()
)
final_chain = keyword_chain | rag_chain

In [42]:
# 실행
query = "연봉 5천만원인 직장인의 소득세는 얼마예요?"
result = final_chain.invoke(query)
print(result)

제공된 문맥만으로는 정확한 소득세액을 계산하기 어렵습니다. 그 이유는 다음과 같습니다.

1. **"종합소득 5천만원"이 과세표준인지 소득금액인지 불분명**합니다.
   - 소득세 계산은 **종합소득과세표준**(소득금액에서 각종 공제를 뺀 금액)을 기준으로 합니다.
   - 과세표준이 5,000만원이라면 제55조 세율표에 따라:
     - 84만원 + (5,000만원 - 1,400만원) × 15% = **624만원** (종합소득산출세액)
   - 그러나 5,000만원이 총급여액이나 종합소득금액이라면, 근로소득공제(제47조), 기본공제(제50조) 등 각종 공제를 적용해 과세표준을 먼저 산출해야 합니다.

2. **필요한 정보가 부족**합니다.
   - 근로소득공제액 계산을 위한 구체적 요율(표)이 문맥에 완전히 제시되지 않았습니다.
   - 기본공제 대상 가족 수, 추가공제, 특별소득공제, 세액공제(근로소득세액공제 제59조, 자녀세액공제 제59조의2 등) 적용 여부를 알 수 없습니다.

**결론적으로**, 과세표준이 5,000만원이라고 가정할 경우 산출세액은 624만원이나, 실제 납부세액은 각종 세액공제를 적용한 후의 금액이므로 제공된 문맥만으로는 정확한 세액을 알 수 없습니다. 정확한 계산을 위해서는 총급여액, 공제 대상 가족 수, 추가 공제·세액공제 적용 여부 등 추가 정보가 필요합니다.


# 9. 답변생성(참조 조항포함)

In [43]:
docs = retriever.invoke(keyword_chain.invoke(query))

def extract_articles_from_docs(documents):
    """
    검색된 문서 documents에서 article 정보를 추출하여 리스트로 반환
    Args:
    - documents : retriever에서 반환된 Document 객체 리스트
    Returns:
        중복 제거된 조항 리스트를 문자로 바꿔서 반환 (ex. '소득세법 제4조, 제60조')
    """
    articles = []
    for doc in documents:
        # metadata에서 article 정보 추출
        article = doc.metadata.get('article', '조항없음')
        article = article.replace('「', '').replace('」', '')
        article_list = article.split(', ')
        articles.extend(article_list)
    # 중복 제거 및 정렬
    unique_article = list(set(articles))
    unique_article.sort(key=lambda x : int(x[:-1]))
    final_article = ['제'+article for article in unique_article if article!='조항없음']
    result = '소득세법 ' + ', '.join(final_article)
    return result
extract_articles_from_docs(docs)

'소득세법 제22조, 제43조, 제47조, 제48조, 제49조, 제50조, 제51조, 제54조, 제55조, 제59조, 제134조'

In [44]:
# 재사용이 가능한 함수 만들기
def ask_with_reference(query:str, k:int=3):
    """
    query의 수정된 query, 답변, 참고조항을 출력하는 함수
    Args:
    - query : 질문
    - k : 검색할 문서의 수(기본값 : 3)
    """
    # query 표준화
    normalized_query = keyword_chain.invoke({"question":query})

    # retriever 생성
    retriever = vectorStore.as_retriever(
        search_type = 'similarity',
        search_kwargs ={
            'k':k,
            'filter':{
                'category':{'$in':categorize_content(normalized_query)}
            }
        }
    )

    # 검색된 문서
    retrieverd_docs = retriever.invoke(normalized_query)

    # 참고 조항 추출
    referenced_articles = extract_articles_from_docs(retrieverd_docs)

    # LLM 답변 생성
    template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 
    모른다고 답하세요.
    질문 : {{query}}
    문맥 : {{context}}
    답변 :"""
    prompt = ChatPromptTemplate.from_template(template)
    prompt_chain = prompt | llm | StrOutputParser()
    answer = prompt_chain.invoke({
        'query':normalized_query,
        'context':format_documents(retrieverd_docs)
    })

    print('='*60)
    print(f'☑️ 원본 질문 : {query}')
    print(f'☑️ 질문 표준화 : {normalized_query}')
    print('='*60)
    print(f'📝 답변 : {answer}\n')
    print(f'📢 참고 조항 : {referenced_articles}')
    print('**위 답변은 AI에 의해 작성된 답변이므로 근소한 차액이 발생할 수 있습니다**')

In [45]:
ask_with_reference('연봉 5000만원인 직장인의 소득세는 얼마예요?')
ask_with_reference('사업소득이 있는 경우 필요경비는 어떻게 계산하나요?')

☑️ 원본 질문 : 연봉 5000만원인 직장인의 소득세는 얼마예요?
☑️ 질문 표준화 : 종합소득 5000만원인 근로소득이 있는 거주자의 소득세는 얼마예요?
📝 답변 : 제공된 문맥(제55조 세율)에 따르면, 거주자의 종합소득에 대한 소득세는 **종합소득과세표준**을 기준으로 계산됩니다. 질문에서 언급된 '종합소득 5000만원'이 **종합소득과세표준**이라고 가정할 경우, 제공된 세율 표를 적용하여 다음과 같이 계산할 수 있습니다.

문맥의 제55조①에 따른 세율표에 의하면, 과세표준이 **1,400만원 초과 5,000만원 이하**인 경우 세액은 다음과 같습니다.

*   **계산식**: 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)
*   **적용**: 84만원 + (5,000만원 - 1,400만원) × 15%
*   **계산**: 84만원 + (3,600만원 × 0.15) = 84만원 + 540만원
*   **결과**: **624만원** (종합소득산출세액)

다만, 이는 소득공제(기본공제, 근로소득공제 등)를 적용하기 전의 과세표준에 세율을 적용한 **산출세액**입니다. 실제 납부세액은 문맥에 언급된 근로소득세액공제(제59조), 자녀세액공제(제59조의2) 등 각종 세액공제와 추가적인 소득공제 여부에 따라 달라질 수 있습니다. 제공된 문맥만으로는 구체적인 공제 내역(총급여액, 가구원 수 등)을 알 수 없어 최종 납부세액을 단정적으로 답변하기는 어렵습니다.

만약 문맥의 세율표를 기준으로 과세표준 5,000만원에 대한 산출세액만을 묻는다면 **624만원**입니다.

📢 참고 조항 : 소득세법 제22조, 제43조, 제47조, 제48조, 제49조, 제50조, 제51조, 제54조, 제55조, 제59조, 제134조
**위 답변은 AI에 의해 작성된 답변이므로 근소한 차액이 발생할 수 있습니다**
☑️ 원본 질문 : 사업소득이 있는 경우 필요경비는 어떻게 계산하나요?
☑️ 질문 표준화 : 사업소득이 있는 경우 필요경비는 어떻게 계산하나요?
📝 답변 : 제공